# Multiagent Pattern - Program Promotion Warm-Up

This warm-up notebook introduces the Multiagent Pattern in a simple non-forensics setting before you move into the Lab 5 case. Instead of asking one agent to do everything, we give different agents different roles and let their outputs build on one another.

If you are new to this course sequence, it may help to review the earlier labs first:
- [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
- [Tool Use Pattern](../lab2_tool_use_pattern/03b_lab_notebook.ipynb)
- [ReAct Pattern](../lab3_react_pattern/03b_lab_notebook.ipynb)
- [Planning Pattern](../lab4_planning_pattern/03a_lab_notebook.ipynb)


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f686c08ca-989b-4083-9128-e6bc2a8c07b5_716x526-3.gif" alt="General Multiagent Pattern" width="500"/>

This figure shows the general Multiagent Pattern: different agents handle different parts of the work, and their outputs are combined into one stronger result.

## What You Will Do
1. Set up the notebook and load the Lab 5 model settings.
2. Read a simple promotion question about the BS Cyber Forensics program at the University of Baltimore.
3. Create three role-specialized agents: `AudienceAgent`, `ProgramValueAgent`, and `OutreachAgent`.
4. Connect the agents so downstream agents receive upstream context automatically.
5. Run the three agents step by step and inspect how context is passed.
6. Run the same workflow again with `Crew`, which keeps the dependency order organized for you.


## Warm-Up Question

Use the following non-forensics question in this notebook:

**How should the University of Baltimore promote its BS Cyber Forensics program over the next three years so it reaches the right students, explains the program's value clearly, and uses realistic outreach methods?**

The three agents in this warm-up have different jobs:
- `AudienceAgent` decides who we should target.
- `ProgramValueAgent` decides what program strengths matter most.
- `OutreachAgent` combines those earlier outputs into one three-year promotion plan.

The main learning goal is not marketing. The goal is to see how multiagent collaboration works when each agent has a clear role.


## Expected Agent Outputs

Before you run the agents, keep these outputs in mind:
- `AudienceAgent` output: priority student groups, why each group fits, and audience advice for the team.
- `ProgramValueAgent` output: most important program strengths, why students should care, and message themes for outreach materials.
- `OutreachAgent` output: the final working three-year promotion plan, including a simple `Year 1`, `Year 2`, and `Year 3` outreach timeline.

So in this warm-up, the first two agents prepare specialized inputs, and `OutreachAgent` writes the combined final three-year plan.


### Step 1: Set Up the Notebook

Run this cell first. It loads the Lab 5 `.env`, checks that the notebook was opened from the correct folder, adds `src/` to the import path, and imports the multiagent classes.

You do not need to memorize every line. The important idea is that this cell prepares the local model settings and the agent classes used in the rest of the notebook.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

LAB_NAME = 'lab5_multiagent_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

# Import the course-specific classes only after src/ has been added to the Python path.
from agentic_patterns.multiagent_pattern.agent import Agent
from agentic_patterns.multiagent_pattern.crew import Crew

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Model:', MODEL)


### Step 2: Create the Shared Promotion Brief

All three agents will work from the same promotion question and the same short program brief. This gives the team a shared starting point before each agent adds its own role-specific thinking.


In [ ]:
PROMOTION_QUESTION = (
    'Create a three-year plan to promote the University of Baltimore BS Cyber Forensics program to prospective students. '
    'Identify the best student audiences, explain the program strengths that matter most, '
    'and recommend realistic outreach methods.'
)

# This shared brief keeps the example grounded and gives every agent the same basic facts.
PROGRAM_BRIEF = """
Program facts:
- The University of Baltimore BS Cyber Forensics program emphasizes hands-on labs, digital evidence analysis, mobile device analysis, and incident response.
- Graduates may pursue careers in digital forensics, cybersecurity operations, and investigative support roles.
- The University of Baltimore wants to attract high school seniors, community-college transfer students, and adult learners changing careers.
- The promotion budget is moderate, so the plan should favor realistic actions rather than expensive campaigns.
- The plan should cover three years, not just one semester.
""".strip()

print(PROMOTION_QUESTION)
print()
print(PROGRAM_BRIEF)


### Step 3: Build the Three Agents

This cell creates the three warm-up agents and connects them with dependencies.

A few plain-language notes:
- `AudienceAgent` goes first and suggests who should be prioritized.
- `ProgramValueAgent` goes second and receives the audience notes as context.
- `OutreachAgent` goes last and receives both earlier outputs before writing the final three-year promotion plan.

The `>>` operator means “send this agent's output to the next agent as context.”


In [ ]:
def build_program_promotion_agents():
    audience_agent = Agent(
        name='AudienceAgent',
        backstory=(
            'You identify the best student audiences for academic program promotion. '
            'You focus on who the program should reach first and why.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Focus only on the audience question. Identify the 2 or 3 target groups that matter most '
            'for this program and explain why each group is a strong fit.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Priority student groups:\n'
            'Why each group fits:\n'
            'Audience advice for the rest of the team:'
        ),
        llm=MODEL,
    )

    program_value_agent = Agent(
        name='ProgramValueAgent',
        backstory=(
            'You explain the strongest value of an academic program in student-friendly language. '
            'You focus on program strengths, benefits, and message themes.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Focus only on program value. Explain which strengths of the University of Baltimore BS Cyber Forensics program '
            'should be highlighted most clearly to prospective students.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Most important program strengths:\n'
            'Why students should care:\n'
            'Message themes for outreach materials:'
        ),
        llm=MODEL,
    )

    outreach_agent = Agent(
        name='OutreachAgent',
        backstory=(
            'You create realistic outreach plans for academic programs. '
            'You combine earlier audience and value insights into one practical phased plan.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Use the earlier agent context to create one practical three-year outreach plan. '
            'Include a simple Year 1, Year 2, and Year 3 timeline.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Priority audience:\n'
            'Main messages:\n'
            'Year 1 outreach focus:\n'
            'Year 2 outreach focus:\n'
            'Year 3 outreach focus:\n'
            'Final working promotion plan:'
        ),
        llm=MODEL,
    )

    # These arrows define the collaboration order.
    audience_agent >> program_value_agent
    audience_agent >> outreach_agent
    program_value_agent >> outreach_agent

    return audience_agent, program_value_agent, outreach_agent


def run_crew_with_markdown(crew: Crew):
    outputs = {}

    # Crew decides the safe dependency order; we only format the outputs for notebook readability.
    for agent in crew.topological_sort():
        output = agent.run()
        outputs[agent.name] = output
        display(Markdown(f'### {agent.name} Output\n\n{output}'))

    return outputs


audience_agent, program_value_agent, outreach_agent = build_program_promotion_agents()

print('AudienceAgent dependents:', audience_agent.dependents)
print('ProgramValueAgent dependencies:', program_value_agent.dependencies)
print('OutreachAgent dependencies:', outreach_agent.dependencies)


### Step 4: Run `AudienceAgent`

This first agent works alone. After it runs, its output is automatically copied into the context of the downstream agents.


In [ ]:
audience_output = audience_agent.run()
display(Markdown('### AudienceAgent Output\n\n' + audience_output))


### Step 5: Inspect the Context Passed to `ProgramValueAgent`

The next cell shows the short-term memory that `ProgramValueAgent` received from `AudienceAgent`. This is one of the most important Multiagent Pattern ideas: downstream agents do not start empty. They inherit useful context from upstream agents.


In [ ]:
display(Markdown('### ProgramValueAgent Received Context\n\n```text\n' + program_value_agent.context + '\n```'))


### Step 6: Run `ProgramValueAgent`

Now the second agent adds a different kind of thinking. It keeps the shared promotion question, but it also sees the earlier audience analysis before writing its own response.


In [ ]:
program_value_output = program_value_agent.run()
display(Markdown('### ProgramValueAgent Output\n\n' + program_value_output))


### Step 7: Run `OutreachAgent`

`OutreachAgent` goes last. It now has context from both earlier agents, so it can combine “who to reach” and “what to say” into one staged three-year outreach plan.


In [ ]:
display(Markdown('### OutreachAgent Received Context\n\n```text\n' + outreach_agent.context + '\n```'))

outreach_output = outreach_agent.run()
display(Markdown('### OutreachAgent Final Promotion Plan\n\n' + outreach_output))


## Part 2: Run the Same Workflow with `Crew`

The manual run above keeps the collaboration logic visible. `Crew` packages the same idea more neatly: it registers the agents, remembers their dependency graph, and lets you execute them in dependency order.

The next two cells create a fresh copy of the same three-agent team inside a `Crew` context so you can see the graph and rerun the workflow cleanly.


In [ ]:
with Crew() as crew:
    crew_audience_agent, crew_program_value_agent, crew_outreach_agent = build_program_promotion_agents()


### Step 8: Visualize the Crew Graph

This graph shows the dependency structure:
- `AudienceAgent` feeds both later agents.
- `ProgramValueAgent` also feeds `OutreachAgent`.
- `OutreachAgent` runs last because it depends on both earlier outputs.


In [ ]:
crew.plot()


### Step 9: Run the Crew End to End

This helper uses `Crew`'s dependency order, but formats the outputs as Markdown so the notebook stays readable.


In [ ]:
crew_outputs = run_crew_with_markdown(crew)


## Why This Warm-Up Matters

This notebook is a non-forensics example, but the collaboration idea is the same one you will use in the main Lab 5 case:
- one agent can focus on the timeline or first interpretation,
- another can verify the evidence,
- another can decide how missing handling records change the final confidence.

Next, move to [03b_lab_notebook.ipynb](./03b_lab_notebook.ipynb) for the full forensic multiagent case.
